# 🚀 T4 GPU LLM Server — Ollama + OpenAI-Compatible `/v1` (Colab Free)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/t4-gpu-llm-api.ipynb)
[![Notebook Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

Runs **Ollama** on a free Tesla T4 and exposes an OpenAI-compatible `/v1` API. No vLLM, no custom FastAPI
server — this is the Ollama runtime, and the notebook says so.

- **Models:** `qwen2.5-coder:7b`, `deepseek-r1:8b`, `llama3.2:3b`, `phi4:14b` (Q4 — 14B is tight on 15 GB)
- **Security:** Ollama ignores the `Authorization` header, so it stays on `127.0.0.1:11434` and only a
  **Basic-auth reverse proxy on `:11435`** is tunnelled. Use the generated session password as the
  API key in any client.
- **Client:** Step 8 downloads `colab_ai_chat.html`; paste the tunnel `/v1` URL and the password.
- **Quota:** ~12 h session, ~60 min idle disconnect on the free tier.


In [ ]:
#@title 🚀 1. Hardware & GPU Diagnostic Check
#@markdown Check available Tesla GPU, CUDA driver, and system RAM.
import subprocess
import os

print("🔍 Analyzing Google Colab Hardware Environment...")
!nvidia-smi --query-gpu=gpu_name,memory.total,memory.free,driver_version --format=csv
!nvcc --version | grep "release"
!free -h

try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
    print(f"\n✅ GPU Detected: {gpu_info}")
    if "T4" in gpu_info:
        print("💡 Tesla T4 detected (15GB VRAM) — Optimal for 7B & 8B Q4 models at full speed!")
except Exception as e:
    print("⚠️ Warning: No GPU detected or nvidia-smi failed. Make sure to select 'T4 GPU' under Runtime > Change runtime type.")


In [ ]:
#@title 📦 2. Install Ollama & Cloudflare Tunnel
#@markdown Installs core system utilities (`zstd`), official **Ollama** engine, and **Cloudflared** tunnel.
import os
import subprocess
from IPython.display import clear_output

print("📥 [1/3] Updating system packages and installing prerequisites (zstd, curl)...\n")
!apt-get update -qq && apt-get install -y -qq zstd curl

print("\n📥 [2/3] Installing Ollama Engine...")
!curl -fsSL https://ollama.ai/install.sh | sh

print("\n📥 [3/3] Downloading & Installing Cloudflared Tunnel Client...")
!curl -fsSL -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb > /dev/null 2>&1
!rm -f cloudflared.deb

print("\n📥 [extra] Installing aiohttp for the auth proxy...")
!pip install -q aiohttp

clear_output()
# Verify installations
ollama_path = subprocess.getoutput("which ollama")
cf_path = subprocess.getoutput("which cloudflared")
print(f"✅ Ollama installed at: {ollama_path}")
print(f"✅ Cloudflared installed at: {cf_path}")
print("🚀 System dependencies ready for GPU server start!")


In [ ]:
#@title ⚡ 3. Start Ollama on localhost only
import os
import socket
import subprocess
import time
from IPython.display import clear_output

subprocess.run(["pkill", "-9", "ollama"], check=False)
time.sleep(1)

server_env = {
    **os.environ,
    "OLLAMA_HOST": "127.0.0.1:11434",   # the tunnel never talks to this port directly
    "OLLAMA_NUM_PARALLEL": "1",         # one slot: 14B Q4 + KV cache must fit in 15 GB
    "OLLAMA_KEEP_ALIVE": "24h",         # keep weights resident in VRAM
}

print("🚀 Starting Ollama on 127.0.0.1:11434 ...")
subprocess.Popen(["ollama", "serve"], env=server_env,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    time.sleep(1)
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", 11434)) == 0:
            clear_output()
            print("✅ Ollama listening on 127.0.0.1:11434 (GPU accelerated)")
            break
else:
    raise RuntimeError("Ollama did not start — rerun this cell and check `ollama serve` output.")


In [ ]:
%%writefile auth_proxy.py
#!/usr/bin/env python3
"""Password-protected reverse proxy for notebook servers (Ollama / ComfyUI).

Colab and Kaggle tunnels expose a port to the public internet. Ollama ignores
the Authorization header entirely and ComfyUI ships without any auth layer, so
an open tunnel means anyone with the URL can use (or reconfigure) your runtime.

This proxy sits on a second port, requires HTTP Basic auth (or
``Authorization: Bearer <password>`` so OpenAI-compatible clients keep working),
and forwards HTTP + WebSocket traffic to the local service.

    python auth_proxy.py --target-port 8188 --port 8189
    PROXY_PASSWORD=... python auth_proxy.py --target-port 11434 --port 11435

Only aiohttp is required (already installed by ComfyUI; one-line pip otherwise).
"""

from __future__ import annotations

import argparse
import asyncio
import base64
import hmac
import os
import sys

from aiohttp import ClientSession, ClientTimeout, TCPConnector, WSMsgType, web

# Headers that must not be forwarded between the two hops.
SKIP_REQUEST = {
    "host", "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",
    "te", "trailer", "transfer-encoding", "upgrade", "expect", "origin", "referer",
}
SKIP_RESPONSE = {
    "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",
    "te", "trailer", "transfer-encoding", "upgrade", "content-encoding", "content-length",
}

SESSION = web.AppKey("session", ClientSession)


def build_app(target_port: int, user: str, password: str) -> web.Application:
    basic = "Basic " + base64.b64encode(f"{user}:{password}".encode()).decode()
    bearer = f"Bearer {password}"
    backend = f"127.0.0.1:{target_port}"

    async def start_session(app: web.Application) -> None:
        # Created on the running loop (required by aiohttp >= 3.10).
        app[SESSION] = ClientSession(
            connector=TCPConnector(limit=128),
            timeout=ClientTimeout(total=None, sock_connect=30, sock_read=None),
        )

    async def close_session(app: web.Application) -> None:
        await app[SESSION].close()

    def authorized(request: web.Request) -> bool:
        supplied = request.headers.get("Authorization", "")
        return hmac.compare_digest(supplied, basic) or hmac.compare_digest(supplied, bearer)

    def reject(request: web.Request) -> web.Response:
        return web.Response(
            status=401,
            text="401: authentication required\n",
            headers={"WWW-Authenticate": 'Basic realm="notebook", charset="UTF-8"'},
        )

    async def proxy_http(request: web.Request) -> web.StreamResponse:
        if not authorized(request):
            return reject(request)
        headers = {k: v for k, v in request.headers.items() if k.lower() not in SKIP_REQUEST}
        body = await request.read() if request.body_exists else None
        try:
            async with request.app[SESSION].request(
                request.method,
                f"http://{backend}{request.rel_url}",
                headers=headers,
                data=body,
                allow_redirects=False,
            ) as upstream:
                response = web.StreamResponse(status=upstream.status, reason=upstream.reason)
                for key, value in upstream.headers.items():
                    if key.lower() not in SKIP_RESPONSE:
                        response.headers[key] = value
                await response.prepare(request)
                if request.method != "HEAD":
                    async for chunk in upstream.content.iter_any():
                        await response.write(chunk)
                await response.write_eof()
                return response
        except Exception as exc:  # backend down / refused
            return web.Response(status=502, text=f"502: backend unavailable ({exc})\n")

    async def proxy_ws(request: web.Request) -> web.StreamResponse:
        if not authorized(request):
            return reject(request)
        client = web.WebSocketResponse(heartbeat=30)
        await client.prepare(request)
        try:
            async with request.app[SESSION].ws_connect(
                f"ws://{backend}{request.rel_url}", heartbeat=30, max_msg_size=0
            ) as upstream:

                async def pump(src, dst) -> None:
                    async for message in src:
                        if message.type == WSMsgType.TEXT:
                            await dst.send_str(message.data)
                        elif message.type == WSMsgType.BINARY:
                            await dst.send_bytes(message.data)
                        elif message.type in (WSMsgType.CLOSE, WSMsgType.CLOSING,
                                              WSMsgType.CLOSED, WSMsgType.ERROR):
                            break

                up = asyncio.create_task(pump(client, upstream))
                down = asyncio.create_task(pump(upstream, client))
                await asyncio.wait({up, down}, return_when=asyncio.FIRST_COMPLETED)
                for task in (up, down):
                    task.cancel()
        except Exception:
            pass
        finally:
            await client.close()
        return client

    app = web.Application(client_max_size=1024 ** 3)
    app.router.add_get("/ws", proxy_ws)
    app.router.add_route("*", "/{tail:.*}", proxy_http)
    app.on_startup.append(start_session)
    app.on_cleanup.append(close_session)
    return app


def main() -> int:
    parser = argparse.ArgumentParser(description="Basic-auth reverse proxy for notebook servers")
    parser.add_argument("--target-port", type=int, required=True, help="local service port (e.g. 8188)")
    parser.add_argument("--port", type=int, default=0, help="listen port (default: target + 1)")
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--user", default=os.getenv("PROXY_USER", "admin"))
    parser.add_argument("--password", default=os.getenv("PROXY_PASSWORD", ""))
    args = parser.parse_args()

    if not args.password:
        print("error: set PROXY_PASSWORD or pass --password", file=sys.stderr)
        return 2

    port = args.port or args.target_port + 1
    print(f"auth proxy: 127.0.0.1:{port} -> 127.0.0.1:{args.target_port} (user={args.user})")
    web.run_app(build_app(args.target_port, args.user, args.password),
                host=args.host, port=port, print=None)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
#@title 🔐 5. Start the Password-Protected Auth Proxy
import os
import secrets
import socket
import subprocess
import sys
import time

if not os.environ.get("STUDIO_PASSWORD"):
    os.environ["STUDIO_PASSWORD"] = secrets.token_urlsafe(9)
PASSWORD = os.environ["STUDIO_PASSWORD"]

log = open("/tmp/auth_proxy.log", "w")
subprocess.Popen(
    [sys.executable, "auth_proxy.py", "--target-port", "11434", "--port", "11435"],
    env={**os.environ, "PROXY_PASSWORD": PASSWORD},
    stdout=log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    time.sleep(0.5)
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", 11435)) == 0:
            break
else:
    raise RuntimeError("Auth proxy did not start — check /tmp/auth_proxy.log")

print("✅ Auth proxy listening on 127.0.0.1:11435 (Basic auth user: admin)")
print(f"🔑 Password for this session: {PASSWORD}")


In [ ]:
#@title 📥 6. Model Selection & Downloader (Best for Free Colab T4)
#@markdown Choose from tested, lag-free models categorized by use case:

MODEL_CATEGORY = "qwen2.5-coder:7b" #@param ["qwen2.5-coder:7b", "qwen2.5-coder:1.5b", "deepseek-r1:8b", "deepseek-r1:14b", "llama3.2:3b", "qwen2.5:7b", "phi4:14b", "mistral:7b", "gemma2:9b"] {allow-input: true}
FORCE_RE_DOWNLOAD = False #@param {type:"boolean"}

import subprocess
import time
from IPython.display import clear_output, Markdown

def is_model_present(model):
    res = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
    return model in res.stdout

def download_model(model, force=False):
    if not force and is_model_present(model):
        print(f"✅ Model '{model}' is already cached and ready to use!")
        return True
        
    print(f"📥 Pulling model: {model} (this may take 1-3 minutes depending on size)...")
    proc = subprocess.Popen(['ollama', 'pull', model], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(proc.stdout.readline, ''):
        if line:
            print(f"\r{line.strip()}", end="", flush=True)
    proc.wait()
    print("")
    return proc.returncode == 0

success = download_model(MODEL_CATEGORY, FORCE_RE_DOWNLOAD)
clear_output()

# Show currently available models
models_out = subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout
print("📋 Currently Pulled & Ready Models on your GPU:")
for line in models_out.strip().split('\n'):
    print(f"   {line}")


In [ ]:
#@title 🌐 7. Expose the Auth Proxy (OpenAI-compatible /v1)
import os
import re
import subprocess
import time
from IPython.display import Markdown, clear_output, display

subprocess.run(["pkill", "-9", "cloudflared"], check=False)
time.sleep(1)

# Log to disk: an unread PIPE blocks cloudflared after 64 KB (kernel pipe buffer).
log_path = "/tmp/cloudflared.log"
if os.path.exists(log_path):
    os.remove(log_path)

log_file = open(log_path, "w")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11435"],
                 stdout=log_file, stderr=subprocess.STDOUT)

public_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists(log_path):
        match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", open(log_path).read())
        if match:
            public_url = match.group(0)
            break

clear_output()
if not public_url:
    raise RuntimeError("Tunnel failed — check /tmp/cloudflared.log")

openai_base = f"{public_url}/v1"
display(Markdown(f"""
### 🚀 API is online — password protected

| Setting | Value |
| :--- | :--- |
| **OpenAI Base URL** | `{openai_base}` |
| **API key** | `{PASSWORD}` (sent as `Authorization: Bearer ...`) |
| **Ollama backend** | `127.0.0.1:11434` (not exposed) |
| **Selected model** | `{MODEL_CATEGORY}` |

```bash
curl {openai_base}/chat/completions \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer {PASSWORD}" \\
  -d '{{"model": "{MODEL_CATEGORY}", "messages": [{{"role": "user", "content": "Hello!"}}]}}'
```

> The trycloudflare URL is random but **public** — the password is what protects it. Rotate it by
> restarting this runtime.
"""))


In [ ]:
#@title 📥 8. Download the Standalone Chat Client
import os
import urllib.request
from IPython.display import Markdown, display

CLIENT_URL = "https://raw.githubusercontent.com/SudhirDevOps1/t4-gpu-llm-api/main/colab_ai_chat.html"
OUTPUT = "colab_ai_chat.html"

try:
    urllib.request.urlretrieve(CLIENT_URL, OUTPUT)
    print(f"✅ Downloaded {OUTPUT} ({os.path.getsize(OUTPUT) / 1024:.0f} KB)")
except Exception as exc:
    print(f"⚠️ Could not download the client: {exc}")
    print("Open it directly in your browser:", CLIENT_URL)

try:
    from google.colab import files
    files.download(OUTPUT)
except Exception:
    pass

display(Markdown(f"""
### 🌟 Chat client ready

1. Open `colab_ai_chat.html` in your browser.
2. Base URL: `{openai_base}` (from step 7).
3. API key: the **session password** printed in step 5.
4. Pick the model you pulled in step 6 and start chatting.
"""))


In [ ]:
#@title 🛠️ 9. Maintenance & Debug Utilities
import subprocess

ACTION = "List Models" #@param ["List Models", "Check Processes (ps)", "Check GPU VRAM (nvidia-smi)", "Stop Ollama", "Stop Auth Proxy", "Stop Tunnel"]

if ACTION == "List Models":
    !ollama list
elif ACTION == "Check Processes (ps)":
    !ps aux | grep -E 'ollama|cloudflared|auth_proxy'
elif ACTION == "Check GPU VRAM (nvidia-smi)":
    !nvidia-smi
elif ACTION == "Stop Ollama":
    !pkill -9 ollama
    print("Ollama stopped.")
elif ACTION == "Stop Auth Proxy":
    !pkill -f auth_proxy.py
    print("Auth proxy stopped (the tunnel will now return 502).")
elif ACTION == "Stop Tunnel":
    !pkill -9 cloudflared
    print("Tunnel stopped.")
